# Hospital Bed Allocation System

In [ ]:
from queue import PriorityQueue
from collections import defaultdict

# CO1: Classes/Dataclasses, State Representation, Problem Formulation
# Patient class to represent individual patients
class Patient:
    def __init__(self, patient_id, name, severity, required_bed, duration_of_stay=1):
        self.patient_id = patient_id # Unique identifier for the patient
        self.name = name             # Name of the patient
        self.severity = severity     # Severity of the patient's condition (e.g., 1-10, 10 being most severe)
        self.required_bed = required_bed # Type of bed required (e.g., "ICU", "General", "Observation")
        self.duration_of_stay = duration_of_stay # Expected duration of stay in days
        self.admission_day = -1      # Day patient was admitted, initialized to -1 (not admitted yet)

    # CO1: Python Functions
    # String representation for Patient objects
    def __str__(self):
        return f"Patient ID: {self.patient_id}, Name: {self.name}, Severity: {self.severity}, Required Bed: {self.required_bed}, Duration: {self.duration_of_stay} days"


# CO1: Classes/Dataclasses, State Representation
# Bed class to represent individual hospital beds
class Bed:
    def __init__(self, bed_id, bed_type):
        self.bed_id = bed_id         # Unique identifier for the bed
        self.bed_type = bed_type     # Type of the bed (e.g., "ICU", "General", "Observation")
        self.is_occupied = False     # Boolean indicating if the bed is currently occupied

    # CO1: Python Functions
    # String representation for Bed objects
    def __str__(self):
        status = "Occupied" if self.is_occupied else "Available"
        return f"Bed ID: {self.bed_id}, Type: {self.bed_type}, Status: {status}"


# CO1: Problem Formulation, State Representation, Classes/Dataclasses, Lists, Dictionaries, Heap/Priority Queue
# Hospital Management System class for bed allocation
class HospitalBedAllocation:
    def __init__(self, initial_beds=None):
        self.beds = []               # List to store all Bed objects in the hospital
        if initial_beds:
            for bed in initial_beds:
                self.add_bed(bed)
        self.patient_queue = PriorityQueue() # PriorityQueue to manage waiting patients by severity
        self.allocations = {}        # Dictionary to store current allocations: bed_id -> Patient object
        self.current_day = 0         # Simulated current day, starting from 0

    # CO1: Python Functions
    # Method to add a bed to the hospital
    def add_bed(self, bed):
        self.beds.append(bed)

    # CO1: Python Functions, Heap/Priority Queue, Utility Functions
    # Method to add a patient to the waiting queue
    def add_patient(self, patient):
        # Patients are prioritized by severity (higher severity = lower negative value = higher priority)
        self.patient_queue.put((-patient.severity, patient))

    # CO1: Python Functions, Rule Sets
    # CO2: Greedy Search
    # CO4: Bounded Rationality
    # Method to allocate a bed to the highest priority patient
    def allocate_bed(self):
        if self.patient_queue.empty():
            print("No patients waiting")
            return

        priority, patient = self.patient_queue.get() # Get the highest priority patient

        # Logic for low-severity patients (less than 5) to be allocated to Observation beds first
        if patient.severity < 5:
            for bed in self.beds:
                if not bed.is_occupied and bed.bed_type == "Observation":
                    bed.is_occupied = True
                    patient.admission_day = self.current_day # Set admission day
                    self.allocations[bed.bed_id] = patient # Store patient object in allocation
                    print("\nBed Allocated Successfully (Observation)")
                    print(patient)
                    print(bed)
                    return
            print(f"No available Observation bed for low-severity patient {patient.name}")

        # Existing logic for other patients (severity >= 5 or no Observation bed available)
        for bed in self.beds:
            if not bed.is_occupied and bed.bed_type == patient.required_bed:
                bed.is_occupied = True
                patient.admission_day = self.current_day # Set admission day
                self.allocations[bed.bed_id] = patient # Store patient object in allocation

                print("\nBed Allocated Successfully")
                print(patient)
                print(bed)
                return

        print(f"No available {patient.required_bed} bed for {patient.name}")

    # CO1: Python Functions
    # CO3: Scheduling/Timetabling
    # CO5: Markov chains & HMM intuition for tracking
    # Method to advance the simulated day and handle discharges
    def advance_day(self):
        self.current_day += 1 # Increment current day
        print(f"\n--- Advancing to Day {self.current_day} ---")
        self.discharge_patients() # Call discharge method

    # CO1: Python Functions
    # CO3: Scheduling/Timetabling
    # CO5: Markov chains & HMM intuition for tracking
    # Method to discharge patients whose duration of stay has ended
    def discharge_patients(self):
        discharged_today = [] # List to temporarily store patients discharged today
        beds_to_free = []     # List to temporarily store bed IDs to be freed

        # Find patients to discharge by iterating through current allocations
        for bed_id, patient in self.allocations.items():
            # Check if patient has been admitted and their duration of stay is complete
            if patient.admission_day != -1 and (self.current_day - patient.admission_day) >= patient.duration_of_stay:
                discharged_today.append(patient)
                beds_to_free.append(bed_id)

        # Process discharges if any patients are to be discharged
        if discharged_today:
            print("\n--- Patients Discharged Today ---")
            for bed_id in beds_to_free:
                patient = self.allocations.pop(bed_id) # Remove patient from allocations
                for bed in self.beds:
                    if bed.bed_id == bed_id:
                        bed.is_occupied = False # Mark bed as free
                        print(f"  Patient {patient.name} (ID: {patient.patient_id}) discharged from Bed {bed.bed_id} ({bed.bed_type}). Bed is now free.")
                        break
        else:
            print("  No patients discharged today.")

    # CO1: Python Functions, Dictionaries
    # Method to display bed summary and details
    def display_beds(self):
        bed_counts = defaultdict(lambda: {'total': 0, 'occupied': 0, 'available_beds': []}) # Dictionary to count beds by type
        for bed in self.beds:
            bed_counts[bed.bed_type]['total'] += 1
            if bed.is_occupied:
                bed_counts[bed.bed_type]['occupied'] += 1
            else:
                bed_counts[bed.bed_type]['available_beds'].append(bed.bed_id)

        summary_lines = [] # List to store summary lines for display
        for bed_type, counts in bed_counts.items():
            available_count = counts['total'] - counts['occupied']
            summary_lines.append(f"  {bed_type}: Total={counts['total']}, Occupied={counts['occupied']}, Available={available_count}")
            if counts['available_beds']:
                summary_lines.append(f"    Available Bed IDs: {', '.join(map(str, counts['available_beds']))}")
            else:
                summary_lines.append(f"    No available beds of type {bed_type}")

        occupied_beds_details = []  # List to store details of occupied beds
        available_beds_details = [] # List to store details of available beds

        # Sort beds by ID for consistent output
        sorted_beds = sorted(self.beds, key=lambda b: b.bed_id)

        for bed in sorted_beds:
            patient_info = ""
            if bed.is_occupied:
                patient_obj = self.allocations.get(bed.bed_id)
                if patient_obj:
                    patient_info = f" (Patient: {patient_obj.name}, Duration: {patient_obj.duration_of_stay} days, Admitted Day: {patient_obj.admission_day})"
                occupied_beds_details.append(f"  Bed ID: {bed.bed_id}, Type: {bed.bed_type}{patient_info}")
            else:
                available_beds_details.append(f"  Bed ID: {bed.bed_type}, Type: {bed.bed_type}")

        # Return structured data for external display
        return {
            'bed_counts': bed_counts,
            'summary_lines': summary_lines,
            'occupied_details': occupied_beds_details,
            'available_details': available_beds_details
        }

    # CO1: Python Functions
    # CO6: Explainable Reasoning Traces
    # Method to display current allocation records
    def display_allocations(self):
        print("\n--- Allocation Records ---")
        # Iterate through beds to find occupied ones and associated patients
        if not self.allocations:
            print("  No beds currently allocated.")
            return

        for bed in self.beds:
            if bed.is_occupied:
                patient = self.allocations.get(bed.bed_id)
                if patient:
                    # Calculate remaining days of stay
                    remaining_days = patient.duration_of_stay - (self.current_day - patient.admission_day)
                    print(f"  Bed {bed.bed_id} ({bed.bed_type}) -> Patient {patient.name} (ID: {patient.patient_id}), Severity: {patient.severity}, Admitted Day: {patient.admission_day}, Remaining Days: {remaining_days})")

### Core Classes for Hospital Bed Allocation System

This section defines the `Patient`, `Bed`, and `HospitalBedAllocation` classes that form the foundation of the system.

###CODE Explaination

1.   **`Patient` Class**:
*   **Purpose**: Represents an individual patient in the system.
*   **Attributes**: Stores `patient_id`, `name`, `severity` (how critical their condition is), `required_bed` (the type of bed they need), `duration_of_stay` (how many days they are expected to stay), and `admission_day` (the day they were admitted, initialized to -1).    
*   **`__str__`** method: Provides a human-readable string representation of a patient object.
2.  **`Bed` Class**:
* **Purpose**: Represents a single hospital bed.
* **Attributes**: Stores `bed_id`, `bed_type` (e.g., 'ICU', 'General', 'Observation'), and `is_occupied` (a boolean indicating if the bed is currently in use).
* **`__str__` method**: Provides a human-readable string representation of a bed object, including its status.

3.    **`HospitalBedAllocation` Class**:
* Purpose: This is the main management class that orchestrates the allocation of beds to patients.
* **Attributes**:
  *  `beds`: A list to store all `Bed` objects in the hospital.
  * `patient_queue`: A `PriorityQueue` from Python's `queue` module. Patients are added here, with higher severity patients receiving higher priority (lower negative severity value).
  * **`allocations`**: A dictionary that maps `bed_id` to the `Patient` object currently occupying that bed.
  * `current_day`: An integer representing the current simulated day.
* **Key Methods**:
   * **`add_bed(self, bed)`**: Adds a new bed to the hospital's list of beds.
   * **`add_patient(self, patient)`**: Adds a patient to the `patient_queue`. Patients are prioritized by severity, so more severe patients are processed first.
    * **`allocate_bed(self)`**: This is a crucial method. It retrieves the highest-priority patient from the queue. It then attempts to find an available bed that matches the patient's `required_bed_type`. It includes special logic for low-severity patients to be allocated to 'Observation' beds first. If a bed is found, it's marked as occupied, the patient's `admission_day` is set, and the allocation is recorded.
     * **`advance_day(self)`**: Increments the c
     `current_day` and calls `discharge_patients()` to process any patients whose stay has ended.
    * **`discharge_patients(self)`**: Iterates through current allocations. If a patient's  `duration_of_stay` has been met (based on `current_day` and `admission_day`), the patient is 'discharged,' their bed is freed (`is_occupied = False`), and they are removed from the `allocations` record.
     * **`display_beds(self)`**: Returns structured data about all beds, including a summary by bed type, and detailed lists of occupied and available beds. This method is designed to provide data for external visualization or reporting rather than direct printing.
     * **`display_allocations(self)`**: Prints a list of all currently occupied beds and the patients assigned to them, including their remaining days of stay.




### Demonstration of Patient Discharge Logic

In [ ]:
# CO1: Problem Formulation (Demonstration of system behavior)
# CO6: Explainable Reasoning Traces (Print statements for tracking flow)

# Initialize a new hospital instance for clear demonstration
demo_hospital = HospitalBedAllocation()

# Add one General bed to the demo hospital
demo_hospital.add_bed(Bed(501, "General"))

# Add a patient with a known short duration of stay for discharge testing
print("\n--- Adding Patient for Discharge Demo ---")
demo_patient = Patient(patient_id=901, name="DischargeTest", severity=5, required_bed="General", duration_of_stay=2)
demo_hospital.add_patient(demo_patient)

# Attempt to allocate a bed to the demo patient
print("\n--- Attempting to Allocate Bed for Demo Patient ---")
demo_hospital.allocate_bed()

# Display current bed status (should show the bed as occupied)
print("\n--- Bed Status After Allocation (Day 0) ---")
bed_status_after_alloc = demo_hospital.display_beds()
for line in bed_status_after_alloc['occupied_details']:
    print(line)
if not bed_status_after_alloc['occupied_details']:
    print("  No beds currently occupied.")

# Advance days until the patient's duration of stay is met to trigger discharge
print(f"\n--- Advancing {demo_patient.duration_of_stay} Days to Trigger Discharge ---")
for _ in range(demo_patient.duration_of_stay):
    demo_hospital.advance_day()

# Display final bed status (should show the bed as available again)
print("\n--- Bed Status After Discharge ---")
bed_status_after_discharge = demo_hospital.display_beds()
for line in bed_status_after_discharge['available_details']:
    print(line)
if not bed_status_after_discharge['available_details']:
    print("  No beds currently available.")

# Display current allocations after discharge (should be empty)
demo_hospital.display_allocations()

### Summary of Patient Discharge Logic

The patient discharge logic in the `HospitalBedAllocation` system ensures that beds are freed up once a patient's expected `duration_of_stay` has been met. This process is managed by the `discharge_patients` method, which is typically called as part of the `advance_day` function.

Here's how it works:

1.  **Daily Check**: When `advance_day()` is called, the `current_day` counter increments, and `discharge_patients()` is invoked.
2.  **Identify Patients for Discharge**: The `discharge_patients()` method iterates through all currently `allocations` (occupied beds).
3.  **Discharge Condition**: For each allocated patient, it checks if their `admission_day` (the day they were admitted) combined with their `duration_of_stay` (expected length of stay) is less than or equal to the `current_day`. If this condition is met, the patient is marked for discharge.
4.  **Freeing the Bed**: Once identified, the patient is removed from the `allocations` dictionary. The corresponding `Bed` object's `is_occupied` status is set back to `False`, making the bed available for new patients.
5.  **Record Keeping**: A message is printed indicating which patient has been discharged from which bed, providing real-time feedback on bed availability.

This mechanism ensures a dynamic and realistic simulation of hospital bed management, where beds become available as patients complete their treatment.

### Demonstration of Hospital Bed Allocation System

This section demonstrates how to initialize the `HospitalBedAllocation` system, add beds and patients, allocate beds, and simulate day advancements with patient discharges.

In [ ]:
import cProfile # CO2: Empirical Profiling, CO6: Performance Engineering
import pstats   # CO2: Empirical Profiling, CO6: Performance Engineering

# CO1: Problem Formulation (Overall system initialization)
# Main Program execution starts here
hospital = HospitalBedAllocation() # Initialize the hospital system

# CO1: State Representation (Adding beds to define initial state)
# Add various types of beds to the hospital (increased quantity and new Observation beds)
hospital.add_bed(Bed(101, "ICU"))
hospital.add_bed(Bed(102, "General"))
hospital.add_bed(Bed(103, "Emergency"))
hospital.add_bed(Bed(104, "ICU"))
hospital.add_bed(Bed(105, "General"))
hospital.add_bed(Bed(106, "Emergency"))
hospital.add_bed(Bed(107, "General"))
hospital.add_bed(Bed(201, "Observation")) # New Observation bed
hospital.add_bed(Bed(202, "Observation")) # Another Observation bed
hospital.add_bed(Bed(203, "Observation")) # Another Observation bed

# CO1: Rule Sets (Severity-based inference)
# CO5: Bayes rule (Intuition - deterministic inference), Uncertainty-aware decisions (deterministic rules)
# Get patient inputs interactively from the user
num_patients = int(input("Enter the number of patients to add: "))
for i in range(num_patients):
    print(f"\nEnter details for Patient {i+1}:")

    while True:
        try:
            patient_id = int(input("  Enter Patient ID: ")) # Get patient ID
            break
        except ValueError:
            print("  Invalid input. Patient ID must be a number.")

    name = input("  Enter Patient Name: ") # Get patient name

    while True:
        try:
            severity = int(input("  Enter Severity (1-10, 10 being most severe): ")) # Get patient severity
            if 1 <= severity <= 10:
                break
            else:
                print("  Severity must be between 1 and 10.")
        except ValueError:
            print("  Invalid input. Severity must be a number.")

    # Infer required_bed and duration_of_stay based on severity rules
    if severity < 5:
        required_bed = "Observation"
        duration_of_stay = 2 # Example: 2 days for Observation bed
        print(f"  Required Bed Type inferred as: {required_bed} (due to severity < 5)")
        print(f"  Duration of Stay inferred as: {duration_of_stay} days")
    elif 5 <= severity <= 7:
        required_bed = "General"
        duration_of_stay = 4 # Example: 4 days for General bed
        print(f"  Required Bed Type inferred as: {required_bed} (due to severity between 5 and 7)")
        print(f"  Duration of Stay inferred as: {duration_of_stay} days")
    else: # severity >= 8
        required_bed = "ICU"
        duration_of_stay = 7 # Example: 7 days for ICU bed
        print(f"  Required Bed Type inferred as: {required_bed} (due to severity >= 8)")
        print(f"  Duration of Stay inferred as: {duration_of_stay} days")

    # Add the created patient to the hospital system
    hospital.add_patient(Patient(patient_id, name, severity, required_bed, duration_of_stay))

# CO2: Greedy Search (Implicit in allocate_bed being called repeatedly)
# CO6: Explainable Reasoning Traces (Output from allocate_bed)
# Attempt to allocate beds for all added patients based on their priority
print("\n--- Attempting to Allocate Beds ---")
for _ in range(num_patients):
    hospital.allocate_bed()

# CO6: Explainable Reasoning Traces (Output of current allocations)
# Display current allocation records
hospital.display_allocations()


# # CO2: Empirical Profiling
# # CO6: Performance Engineering
# # Profile the allocate_bed method (commented out for normal execution)
# print("\n--- Profiling allocate_bed method ---")
# with cProfile.Profile() as pr:
#     hospital.allocate_bed()

# stats = pstats.Stats(pr)
# stats.sort_stats(pstats.SortKey.TIME)
# stats.print_stats()


### Hospital Bed Overview - Summary by Bed Type

In [ ]:
# CO6: Explainable Reasoning Traces
# Display a general overview of hospital beds
print("\n--- Hospital Bed Overview ---")
bed_status = hospital.display_beds() # Get structured bed status data

# Print the summary by bed type
print("\n--- Summary by Bed Type ---")
for line in bed_status['summary_lines']:
    print(line)

### Hospital Bed Overview - Occupied Beds Details

In [ ]:
# CO6: Explainable Reasoning Traces
# Display detailed status of occupied beds
print("\n--- Detailed Bed Status (Occupied) ---")
bed_status = hospital.display_beds() # Get structured bed status data
if bed_status['occupied_details']:
    for line in bed_status['occupied_details']:
        print(line)
else:
    print("  No beds currently occupied.")

### Hospital Bed Overview - Available Beds Details

In [ ]:
# CO6: Explainable Reasoning Traces
# Display detailed status of available beds
print("\n--- Detailed Bed Status (Available) ---")
bed_status = hospital.display_beds() # Get structured bed status data
if bed_status['available_details']:
    for line in bed_status['available_details']:
        print(line)
else:
    print("  No beds currently available.")

### How the System Works

1. Beds are added to the hospital.
2. Patients are inserted into a priority queue.
3. Patients with higher severity get priority.
4. The system checks available beds.
5. Matching beds are allocated.
6. Allocation records are stored.


## AI Topics in Hospital Bed Allocation System

This table outlines key concepts from Artificial Intelligence (AI) that are currently applied or implicitly present within the Hospital Bed Allocation System, based on the provided context.

| CO  | Topic Name                                        | Where It Is Used in Hospital Bed Allocation System      | Why It Is Used                                                                                                                                                             |
| --- | ------------------------------------------------- | ------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| CO1 | Python Functions                                  | Allocation modules (`add_bed`, `add_patient`, `allocate_bed`, etc.) | To organize program logic and modularize tasks.                                                                                                                            |
| CO1 | Classes/Dataclasses                               | `Patient`, `Bed`, `HospitalBedAllocation` classes       | To store structured data and define system components.                                                                                                                     |
| CO1 | Lists                                             | `self.beds` in `HospitalBedAllocation`                  | To maintain records of all beds.                                                                                                                                           |
| CO1 | Dictionaries                                      | `self.allocations` (`bed_id` to `Patient` mapping)      | To map bed IDs with occupied patients, and `defaultdict` for bed counts.                                                                                                   |
| CO1 | Heap/Priority Queue                               | `self.patient_queue` in `HospitalBedAllocation`         | To prioritize emergency patients and allocate beds based on severity.                                                                                                      |
| CO1 | Problem Formulation                               | Overall `HospitalBedAllocation` system design           | To define states (bed status, patient queues), actions (adding beds/patients, allocating beds), and goals (efficient bed management).                                         |
| CO1 | State Representation                              | `beds` list, `allocations` dictionary, `is_occupied` flag, `current_day` | To explicitly represent the real-time status of the hospital system.                                                                                                       |
| CO1 | Rule Sets                                         | Logic within `allocate_bed` (e.g., low-severity to 'Observation'), inferred `required_bed`/`duration_of_stay` | To apply hospital policies and rules automatically based on patient severity.                                                                                              |
| CO2 | Empirical Profiling                               | `cProfile` and `pstats` imports (commented out in `89cc5419`) | To allow for measurement and optimization of the system's runtime and memory usage.                                                                                        |
| CO2 | Greedy Search                                     | `allocate_bed` method                                   | To iterate through available beds and allocate the first suitable one found, aiming for a fast decision.                                                                   |
| CO3 | Scheduling/Timetabling                            | `advance_day` and `discharge_patients` methods          | To manage patient admissions and discharges based on `duration_of_stay` as the simulated days progress.                                                                    |
| CO4 | Utility Functions                                 | Patient `severity` in `add_patient` and `allocate_bed`  | To implicitly prioritize actions that yield higher 'value', such as treating more severe patients first.                                                                   |
| CO4 | Bounded Rationality                               | Simplified rules and sequential bed search in `allocate_bed` | To make practical, quick decisions within operational constraints without complex, exhaustive optimization.                                                                |
| CO5 | Probability review                                | Patient `severity` and `duration_of_stay` attributes    | To represent a simplified, deterministic approach to patient conditions and needs, laying conceptual groundwork for probability modeling.                                  |
| CO5 | Bayes rule (Intuition)                            | Inference of `required_bed` and `duration_of_stay` based on `severity` | To act as deterministic rules for conditional inference where patient attributes lead to specific requirements.                                                            |
| CO5 | Markov chains & HMM intuition for tracking        | `advance_day` and `discharge_patients` methods managing patient states over time | To provide a foundational structure for modeling future patient state transitions and bed availability.                                                                    |
| CO5 | Uncertainty-aware decisions (expected utility concept) | Deterministic rules for bed allocation based on patient `severity` | To simplify decision-making in medical scenarios, laying a conceptual basis for considering expected outcomes under probabilistic conditions.                          |
| CO6 | Hybrid Architectures                              | Combination of rule-based logic and search logic        | To leverage different AI techniques for effective patient management and bed allocation (e.g., severity-based rules and greedy bed search).                              |
| CO6 | Explainable Reasoning Traces                      | Extensive `print` statements in core methods            | To provide transparency and understanding of the system's operational flow and decisions (e.g., allocation and discharge messages).                                         |
| CO6 | Performance Engineering                           | `cProfile` and `pstats` imports (commented out in `89cc5419`) | To allow for measurement and optimization of the system's runtime and memory usage.                                                                                        |
| CO6 | Ethics/Limitations                                | Prioritization by patient `severity`                    | To implement a policy that implicitly addresses ethical considerations by prioritizing critical patient care.                                                              |
| CO6 | Readiness for ML/DL/NLP/GenAI                     | Object-oriented design with classes and data structures | To provide a modular and extensible foundation for future integration of advanced AI systems.                                                                              |

## Future Improvements

You can further add:

* Login system
* Database connection
* GUI using Tkinter
* Web app using Flask/Django
* A* search for nearest hospital
* Bayesian prediction for ICU requirement
* Machine learning for patient risk prediction